# Landscape-size diagnostic — PARALLEL fan-out (2026-05-30)

**Drill-down, NOT a graduation run.** Same diagnostic as the serial notebook, but
fanned out so it actually uses the GPU.

**Why a 90 GB GPU alone doesn't help:** this workload is GPU-*light* (D=4096 ops,
sync-bound, ~0.5 GB) — one process leaves the GPU ~99% idle. A bigger GPU doesn't
speed up one process. **Parallelism does:** this notebook runs **3 L × 10 seeds =
30 single-cell CUDA workers** concurrently (the project's blessed fan-out pattern),
filling the idle GPU. It also runs only arms **A** (consolidated) + **C** (frozen)
— all the pre-committed read needs — cutting work ~2.5× more. Net: **~5–10 min**
instead of ~1.8 hr, and bit-identical results (cells are order-independent, verified).

- Pre-committed read: `notes/notes/2026-05-30-landscape-sweep-diagnostic-precommit.md`
- Background: `reports/117_frameb_leveldid_feasibility_consolidation_variance.md`

**Before running:** *Runtime → Change runtime type → GPU* (A100/L4). Run top-to-bottom;
the fan-out cell prints the table + verdict.


In [ ]:
# 1. Clone + checkout the branch, verify the parallel diagnostic code is present.
import os
from pathlib import Path
REPO_DIR = '/content/Neuro-AI'
BRANCH = 'codex/phase5-prime-bundle-first-scene-memory'
%cd /content
!rm -rf Neuro-AI
!git clone https://github.com/Dypatterson/Neuro-AI.git
%cd Neuro-AI
!git checkout {BRANCH}
!git log --oneline -3
os.chdir(REPO_DIR)
for rel in ('experiments/gate0_frame_a.py', 'scripts/landscape_sweep_parallel.py',
            'scripts/landscape_sweep_read.py'):
    if not (Path(REPO_DIR) / rel).exists():
        raise SystemExit(f'{rel} missing — push this session first.')
if '--landscape-size' not in (Path(REPO_DIR)/'experiments'/'gate0_frame_a.py').read_text():
    raise SystemExit('--landscape-size flag missing — wrong branch?')
print('Parallel landscape diagnostic verified on branch.')


In [ ]:
# 2. Pre-warm the WikiText-2 cache so the 30 workers share ONE on-disk copy
#    (instead of 30 simultaneous downloads). CPU-only; no CUDA touched.
import os
os.environ['PYTHONPATH'] = '/content/Neuro-AI/src'
!cd /content/Neuro-AI && PYTHONPATH=/content/Neuro-AI/src python -c "import sys; sys.path[:0]=['experiments','src']; import c3_phase3_exit_criterion as c3; from pathlib import Path; c=c3._load_wikitext_corpus(repo_root=Path('.'), wikitext_name='wikitext-2-raw-v1', vocab_cap=1000); print('wikitext cached: vocab', c.vocab_size, '| train_ids', len(c.train_ids))"


In [ ]:
# 3. SMOKE — tiny synthetic fan-out on CUDA: confirms the parallel pipeline works.
!cd /content/Neuro-AI && PYTHONPATH=/content/Neuro-AI/src python scripts/landscape_sweep_parallel.py --smoke --device cuda --out reports/landscape_smoke


In [ ]:
# 4. THE FAN-OUT RUN — 3 L x 10 seeds = 30 single-cell CUDA workers (arms A + C).
#    Parent stays CPU-only; each worker owns a CUDA context. ~5-10 min.
#    (Raise --max-concurrent if your runtime has many vCPUs; lower it if it thrashes.)
!cd /content/Neuro-AI && PYTHONPATH=/content/Neuro-AI/src python scripts/landscape_sweep_parallel.py --device cuda --landscapes 64,256,512 --seeds 0-9 --out reports/landscape_2026-05-30 --max-concurrent 16


In [ ]:
# 5. Persist results to Drive.
from google.colab import drive; drive.mount('/content/drive')
import shutil
shutil.copytree('/content/Neuro-AI/reports/landscape_2026-05-30',
                '/content/drive/MyDrive/neuro-ai/results/landscape_2026-05-30',
                dirs_exist_ok=True)
print('copied to Drive.')


## What the verdict means (pre-registered — not post-hoc)

| verdict | meaning | next |
|---|---|---|
| **VARIANCE-REDUCIBLE** (σ_A(512) ≤ 0.10 AND mean(A−C) ≥ 0.020) | a bigger landscape tames the consolidation-injected variance | propose a powered run at the best L (op-point sign-off); level-DiD n drops ~424 → ~100–187, slope worth revisiting |
| **VARIANCE-IRREDUCIBLE** (σ_A(512) ≥ 0.13) | landscape is not the knob | Frame B mechanism / operating-point rethink — not more compute |
| **PARTIAL** (0.10 < σ_A(512) < 0.13) | ambiguous | read the σ_A(L) curve; consider L=1024 or modest L↑ + n↑ |
| **CAPACITY-WALL** (mean(A−C) < 0.015 at any L) | bigger landscape exceeded Hopfield capacity at β=10 and killed the signal | bigger L is off the table even if σ dropped |

σ_A(L=64) **must reproduce ≈ 0.157** (the recovered run) — it's the reproducibility anchor (the harness warns if it lands outside 0.13–0.18). If it doesn't, stop: environment/repro problem.

**Paste the printed table + verdict back to the assistant** (or share the Drive folder `landscape_2026-05-30/`) for the full write-up (Report 118) and the next decision.
